# Notebook 07 — Adaptive Execution Selection

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebooks 01–06 built the pipeline:

1. input-distribution structure  
2. cache / branching proxies  
3. SIMD vs scalar execution paths  
4. constraint phase maps  
5. benchmark ingestion  
6. hardware-counter overlays  

Notebook 07 turns those layers into an **adaptive execution selector**.

Constraint view:
> execution is a constraint-selection problem: distribution structure → execution topology → hardware pressure → adaptive path choice.

## Goals

1. Load Notebook 06 hardware-counter overlays.
2. Build rule-based execution-regime scores.
3. Predict:
   - scalar path
   - SIMD path
   - coherent-local path
   - fragmented-irregular path
   - hybrid path
4. Compare predicted execution regime against observed behavior.
5. Estimate adaptive-selection improvement.
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 06 overlay table

If Notebook 06 output is not available, this notebook uses a fallback table matching the same schema.

In [ ]:
overlay_path = RESULTS_DIR / "notebook06_hardware_counter_overlays.csv"

if overlay_path.exists():
    df = pd.read_csv(overlay_path)
    print("Loaded:", overlay_path)
else:
    print("Notebook 06 output not found; using fallback overlay table.")
    df = pd.DataFrame([
        {
            "distribution": "low_entropy_repeating",
            "regime": "coherent-local",
            "observed_throughput_mib_s": 1650,
            "observed_latency_ns": 0.60,
            "coherence_score": 0.75,
            "fragmentation_score": 0.02,
            "simd_suitability": 0.26,
            "scalar_suitability": 0.98,
            "branch_miss_rate": 0.0005,
            "cache_miss_rate": 0.005,
            "ipc": 2.0,
            "counter_pressure_score": 0.04,
        },
        {
            "distribution": "sequential_ids",
            "regime": "scalar-favorable",
            "observed_throughput_mib_s": 1350,
            "observed_latency_ns": 0.74,
            "coherence_score": 0.52,
            "fragmentation_score": 0.36,
            "simd_suitability": 0.39,
            "scalar_suitability": 0.62,
            "branch_miss_rate": 0.0048,
            "cache_miss_rate": 0.012,
            "ipc": 1.75,
            "counter_pressure_score": 0.24,
        },
        {
            "distribution": "uniform_32bit",
            "regime": "simd-favorable",
            "observed_throughput_mib_s": 1900,
            "observed_latency_ns": 0.53,
            "coherence_score": 0.24,
            "fragmentation_score": 0.95,
            "simd_suitability": 0.62,
            "scalar_suitability": 0.08,
            "branch_miss_rate": 0.028,
            "cache_miss_rate": 0.022,
            "ipc": 2.6,
            "counter_pressure_score": 0.41,
        },
        {
            "distribution": "zipfian_smallints",
            "regime": "simd-favorable",
            "observed_throughput_mib_s": 1500,
            "observed_latency_ns": 0.67,
            "coherence_score": 0.42,
            "fragmentation_score": 0.79,
            "simd_suitability": 0.67,
            "scalar_suitability": 0.22,
            "branch_miss_rate": 0.031,
            "cache_miss_rate": 0.019,
            "ipc": 1.85,
            "counter_pressure_score": 0.51,
        },
        {
            "distribution": "clustered_ranges",
            "regime": "fragmented-irregular",
            "observed_throughput_mib_s": 950,
            "observed_latency_ns": 1.05,
            "coherence_score": 0.22,
            "fragmentation_score": 1.00,
            "simd_suitability": 0.45,
            "scalar_suitability": 0.05,
            "branch_miss_rate": 0.047,
            "cache_miss_rate": 0.039,
            "ipc": 1.28,
            "counter_pressure_score": 1.00,
        },
    ])

df.head()

## Normalize feature columns

The selector combines structural, execution-path, benchmark, and counter information.

In [ ]:
work = df.copy()

def norm01(series):
    s = pd.Series(series).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

required_defaults = {
    "coherence_score": 0.0,
    "fragmentation_score": 0.0,
    "simd_suitability": 0.0,
    "scalar_suitability": 0.0,
    "counter_pressure_score": 0.0,
    "branch_miss_rate": 0.0,
    "cache_miss_rate": 0.0,
    "ipc": 1.0,
    "observed_throughput_mib_s": 0.0,
    "observed_latency_ns": 0.0,
}

for col, default in required_defaults.items():
    if col not in work.columns:
        work[col] = default
    work[col] = pd.to_numeric(work[col], errors="coerce").fillna(default)

work["throughput_norm"] = norm01(work["observed_throughput_mib_s"])
work["latency_norm"] = norm01(work["observed_latency_ns"])
work["ipc_norm"] = norm01(work["ipc"])
work["branch_miss_norm"] = norm01(work["branch_miss_rate"])
work["cache_miss_norm"] = norm01(work["cache_miss_rate"])

work["hardware_pressure"] = (
    0.35 * work["counter_pressure_score"] +
    0.25 * work["branch_miss_norm"] +
    0.25 * work["cache_miss_norm"] +
    0.15 * (1.0 - work["ipc_norm"])
).clip(0, 1)

work.head()

## Define adaptive execution-regime scores

Each regime is scored as a transparent rule-based model:

- **scalar_path_score** favors scalar suitability, coherence, low pressure.
- **simd_path_score** favors SIMD suitability, IPC, throughput.
- **coherent_local_score** favors coherence, low fragmentation, low pressure.
- **fragmented_irregular_score** identifies high fragmentation and high counter pressure.
- **hybrid_score** identifies mixed signals where no single path dominates.

In [ ]:
work["scalar_path_score"] = (
    0.45 * work["scalar_suitability"] +
    0.25 * work["coherence_score"] +
    0.20 * (1.0 - work["hardware_pressure"]) +
    0.10 * (1.0 - work["latency_norm"])
).clip(0, 1)

work["simd_path_score"] = (
    0.40 * work["simd_suitability"] +
    0.25 * work["ipc_norm"] +
    0.25 * work["throughput_norm"] +
    0.10 * (1.0 - work["cache_miss_norm"])
).clip(0, 1)

work["coherent_local_score"] = (
    0.45 * work["coherence_score"] +
    0.25 * (1.0 - work["fragmentation_score"]) +
    0.20 * (1.0 - work["hardware_pressure"]) +
    0.10 * work["scalar_suitability"]
).clip(0, 1)

work["fragmented_irregular_score"] = (
    0.40 * work["fragmentation_score"] +
    0.35 * work["hardware_pressure"] +
    0.15 * work["branch_miss_norm"] +
    0.10 * work["cache_miss_norm"]
).clip(0, 1)

# Hybrid: high when scalar/SIMD scores are close and pressure is moderate.
score_gap = (work["simd_path_score"] - work["scalar_path_score"]).abs()
work["hybrid_score"] = (
    0.45 * (1.0 - score_gap) +
    0.25 * (1.0 - (work["hardware_pressure"] - 0.5).abs() * 2).clip(0, 1) +
    0.15 * work["coherence_score"] +
    0.15 * work["throughput_norm"]
).clip(0, 1)

regime_score_cols = [
    "scalar_path_score",
    "simd_path_score",
    "coherent_local_score",
    "fragmented_irregular_score",
    "hybrid_score",
]

score_to_label = {
    "scalar_path_score": "select_scalar",
    "simd_path_score": "select_simd",
    "coherent_local_score": "select_coherent_local",
    "fragmented_irregular_score": "select_guarded_fallback",
    "hybrid_score": "select_hybrid",
}

work["predicted_selector"] = work[regime_score_cols].idxmax(axis=1).map(score_to_label)
work["selector_confidence"] = (
    work[regime_score_cols].max(axis=1) -
    work[regime_score_cols].apply(lambda r: r.sort_values().iloc[-2], axis=1)
)

work[["distribution", "predicted_selector", "selector_confidence"] + regime_score_cols]

## Infer observed best-behavior label

This is a simple observed label derived from available measurements:

- high throughput + high IPC → observed SIMD-like
- low pressure + high coherence → observed coherent-local
- high pressure + low throughput → observed guarded fallback
- scalar suitability + low branch miss → observed scalar-like
- otherwise hybrid

In [ ]:
def observed_label(row):
    if row["hardware_pressure"] > 0.75 and row["throughput_norm"] < 0.35:
        return "select_guarded_fallback"
    if row["throughput_norm"] > 0.75 and row["ipc_norm"] > 0.75:
        return "select_simd"
    if row["coherence_score"] > 0.65 and row["hardware_pressure"] < 0.25:
        return "select_coherent_local"
    if row["scalar_suitability"] > 0.55 and row["branch_miss_norm"] < 0.35:
        return "select_scalar"
    return "select_hybrid"

work["observed_behavior_label"] = work.apply(observed_label, axis=1)
work["selector_matches_observed"] = work["predicted_selector"] == work["observed_behavior_label"]

work[["distribution", "predicted_selector", "observed_behavior_label", "selector_matches_observed"]]

## Estimate adaptive-selection improvement

This estimates potential improvement from avoiding poorly matched paths.

It is a proxy, not a benchmark claim.

In [ ]:
# Penalty if predicted selector disagrees with observed behavior.
work["mismatch_penalty"] = np.where(work["selector_matches_observed"], 0.0, 0.15 + 0.25 * (1.0 - work["selector_confidence"]))

# Improvement opportunity is larger when pressure is high or mismatch exists.
work["adaptive_improvement_opportunity"] = (
    0.45 * work["hardware_pressure"] +
    0.35 * work["fragmentation_score"] +
    0.20 * work["mismatch_penalty"]
).clip(0, 1)

# Conservative estimated gain in percent.
work["estimated_adaptive_gain_pct"] = 100.0 * (
    0.05 + 0.25 * work["adaptive_improvement_opportunity"]
)

work[[
    "distribution",
    "predicted_selector",
    "observed_behavior_label",
    "adaptive_improvement_opportunity",
    "estimated_adaptive_gain_pct"
]]

## Export selector table

In [ ]:
csv_path = RESULTS_DIR / "notebook07_adaptive_execution_selection.csv"
json_path = RESULTS_DIR / "notebook07_adaptive_execution_selection.json"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — Adaptive selector scores by distribution

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook07_selector_scores.png"

score_df = work.set_index("distribution")[regime_score_cols]

plt.figure(figsize=(11, 5))
bottom = np.zeros(len(score_df))
x = np.arange(len(score_df.index))
for col in regime_score_cols:
    plt.bar(x, score_df[col], bottom=bottom, label=col.replace("_score", ""))
    bottom += score_df[col].values

plt.xticks(x, score_df.index, rotation=45, ha="right")
plt.ylabel("Stacked selector scores")
plt.title("Adaptive Execution Selection: Regime Scores")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Selector phase map

X-axis: hardware pressure.  
Y-axis: coherence.  
Point size: throughput.  
Label: predicted selector.

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook07_selector_phase_map.png"

sizes = 120 + 500 * work["throughput_norm"]
plt.figure(figsize=(9, 6))
plt.scatter(work["hardware_pressure"], work["coherence_score"], s=sizes, alpha=0.75)
for _, row in work.iterrows():
    label = f"{row['distribution']}\n{row['predicted_selector'].replace('select_', '')}"
    plt.annotate(label, (row["hardware_pressure"], row["coherence_score"]), fontsize=8)
plt.xlabel("Hardware pressure")
plt.ylabel("Coherence score")
plt.title("Adaptive Selector Phase Map")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Predicted vs observed behavior

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook07_predicted_vs_observed.png"

comparison = work[["distribution", "predicted_selector", "observed_behavior_label", "selector_matches_observed"]].copy()
comparison["match_numeric"] = comparison["selector_matches_observed"].astype(int)

plt.figure(figsize=(9, 5))
plt.bar(comparison["distribution"], comparison["match_numeric"])
plt.xticks(rotation=45, ha="right")
plt.yticks([0, 1], ["mismatch", "match"])
plt.ylabel("Selector agreement")
plt.title("Adaptive Selector: Predicted vs Observed Behavior")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Adaptive improvement opportunity

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook07_adaptive_improvement_opportunity.png"

plot_df = work.sort_values("adaptive_improvement_opportunity")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["distribution"], plot_df["adaptive_improvement_opportunity"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Improvement opportunity")
plt.title("Adaptive Execution: Improvement Opportunity")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Policy table heatmap

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook07_policy_matrix.png"

matrix_cols = [
    "scalar_path_score",
    "simd_path_score",
    "coherent_local_score",
    "fragmented_irregular_score",
    "hybrid_score",
    "hardware_pressure",
    "adaptive_improvement_opportunity",
]

mat = work.set_index("distribution")[matrix_cols].sort_values("adaptive_improvement_opportunity", ascending=False)

plt.figure(figsize=(10, 5))
plt.imshow(mat.values, aspect="auto")
plt.yticks(range(len(mat.index)), mat.index)
plt.xticks(range(len(matrix_cols)), matrix_cols, rotation=45, ha="right")
plt.colorbar(label="Normalized score")
plt.title("Adaptive Execution Policy Matrix")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_07_adaptive_execution_selection.md"

summary_cols = [
    "distribution",
    "predicted_selector",
    "observed_behavior_label",
    "selector_matches_observed",
    "selector_confidence",
    "coherence_score",
    "hardware_pressure",
    "observed_throughput_mib_s",
    "adaptive_improvement_opportunity",
    "estimated_adaptive_gain_pct",
]

lines = [
    "# Report 07 — Adaptive Execution Selection",
    "",
    "This report turns RML structural, benchmark, and hardware-counter layers into an adaptive execution-selector model.",
    "",
    "Constraint view:",
    "> execution is a constraint-selection problem: distribution structure → execution topology → hardware pressure → adaptive path choice.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    "",
    "## Adaptive selector summary",
    "",
    work[summary_cols].to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- The selector treats scalar, SIMD, coherent-local, guarded-fallback, and hybrid execution as regimes.",
    "- Selector confidence measures how clearly one regime dominates the alternatives.",
    "- Mismatches identify where the rule-based policy needs measured benchmark correction.",
    "- Improvement opportunity highlights distributions where adaptive routing could matter most.",
    "",
    "## Next step",
    "",
    "Notebook 08 can simulate online runtime adaptation: classify distribution windows, choose execution paths, and estimate throughput under switching costs.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook07_adaptive_execution_selection_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook07_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_07_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))